In [2]:
import random
import math
from itertools import product

def evaluar_clausula(clausula, asignacion):
  for lit in clausula:
    var = abs(lit)
    valor = asignacion[var]
    if (lit > 0 and valor) or (lit < 0 and not valor):
      return True
  return False


def evaluar_formula(formula, asignacion):
  return sum(evaluar_clausula(c, asignacion) for c in formula)


def random_assignment(variables, formula):
  asignacion = {v: random.choice([True, False]) for v in variables}
  return asignacion, evaluar_formula(formula, asignacion)


def derandomized_max3sat(variables, formula):
  asignacion = {}
  vars_lista = list(variables)

  for var in vars_lista:
      mejor_val, mejor_score = None, -1
      for val in [True, False]:
        asignacion[var] = val
        score = _expected_sat(formula, asignacion, vars_lista)
        if score > mejor_score:
          mejor_score = score
          mejor_val = val
      asignacion[var] = mejor_val

  return asignacion, evaluar_formula(formula, asignacion)


def _expected_sat(formula, fijadas, todas):
  libres = [v for v in todas if v not in fijadas]
  total = 0.0
  for clausula in formula:
      vars_libres_en_c = [abs(lit) for lit in clausula if abs(lit) in libres]
      sat_fijada = any(
          (lit > 0 and fijadas.get(abs(lit)) == True) or
          (lit < 0 and fijadas.get(abs(lit)) == False)
          for lit in clausula if abs(lit) in fijadas
      )
      if sat_fijada:
        total += 1.0
      elif vars_libres_en_c:
        k = len(vars_libres_en_c)
        total += 1.0 - (1.0 / (2 ** k))
  return total



def random_walk_sat(variables, formula, max_flips=None, restarts = 10):
    n = len(variables)
    if max_flips is None:
      max_flips = 3*n*int(math.log2(n + 2))

    vars_lista = list(variables)
    mejor_asig = None
    mejor_score = -1

    for _ in range(restarts):
      asig = {v: random.choice([True, False]) for v in vars_lista}

      for _ in range(max_flips):
        score = evaluar_formula(formula,asig)
        if score > mejor_score:
          mejor_score = score
          mejor_asig = dict(asig)
        if score == len(formula):
          return mejor_asig, mejor_score

        insatisfechas = [c for c in formula
                         if not evaluar_clausula(c, asig)]
        clausula = random.choice(insatisfechas)

        var = abs(random.choice(clausula))
        asig[var] = not asig[var]

    return mejor_asig, mejor_score

def approx_max3sat(variables, formula, iteraciones = 200):
  mejor_asig = None
  mejor_score = -1
  for _ in range(iteraciones):
    asig, score = random_assignment(variables, formula)
    if score > mejor_score:
      mejor_score = score
      mejor_asig  = dict(asig)
    if mejor_score == len(formula):
      break

  asig_rw, score_rw = random_walk_sat(variables, formula)
  if score_rw > mejor_score:
    mejor_score = score_rw
    mejor_asig = asig_rw
  return mejor_asig, mejor_score


if __name__ == "__main__":
    # Variables: {1, 2, 3, 4, 5}
    # Fórmula: (x1 ∨ ¬x2 ∨ x3) ∧ (¬x1 ∨ x2 ∨ x4) ∧ (x2 ∨ ¬x3 ∨ x5)
    #        ∧ (¬x1 ∨ ¬x4 ∨ x5) ∧ (x1 ∨ x3 ∨ ¬x5)
    variables = {1, 2, 3, 4, 5}
    formula = [
        [1, -2,  3],
        [-1,  2,  4],
        [2, -3,  5],
        [-1, -4,  5],
        [1,  3, -5],
    ]

    print("=== 1. Asignación aleatoria pura ===")
    asig, score = random_assignment(variables, formula)
    print(f"  Asignación: {asig}")
    print(f"  Satisfechas: {score}/{len(formula)}  "
          f"(garantía esperanza ≥ {7/8*len(formula):.2f})\n")

    print("=== 2. Derandomización (esperanza condicional) ===")
    asig, score = derandomized_max3sat(variables, formula)
    print(f"  Asignación: {asig}")
    print(f"  Satisfechas: {score}/{len(formula)}  "
          f"(garantía determinista ≥ {7/8*len(formula):.2f})\n")

    print("=== 3. Random Walk Local Search ===")
    asig, score = random_walk_sat(variables, formula)
    print(f"  Asignación: {asig}")
    print(f"  Satisfechas: {score}/{len(formula)}\n")

    print("=== 4. Algoritmo combinado ===")
    asig, score = approx_max3sat(variables, formula)
    print(f"  Asignación: {asig}")
    print(f"  Satisfechas: {score}/{len(formula)}")

=== 1. Asignación aleatoria pura ===
  Asignación: {1: False, 2: False, 3: False, 4: True, 5: True}
  Satisfechas: 4/5  (garantía esperanza ≥ 4.38)

=== 2. Derandomización (esperanza condicional) ===
  Asignación: {1: True, 2: True, 3: True, 4: False, 5: True}
  Satisfechas: 5/5  (garantía determinista ≥ 4.38)

=== 3. Random Walk Local Search ===
  Asignación: {1: True, 2: True, 3: False, 4: True, 5: True}
  Satisfechas: 5/5

=== 4. Algoritmo combinado ===
  Asignación: {1: True, 2: True, 3: False, 4: False, 5: True}
  Satisfechas: 5/5
